# S09 · Codificación de variables categóricas

NovaMarket · Preprocesamiento de Datos · Ricardo Borja

---

Codificación de todas las columnas categóricas del dataset con la técnica que corresponde a cada
una según su naturaleza (nominal u ordinal) y su cardinalidad, ajustando cualquier codificador que
aprenda de los datos únicamente con el conjunto de entrenamiento para evitar fuga de datos.

Insumo: `S08_PD_Grupo05_DatasetEscalado.csv`, salida de la sesión de escalado.
Salida: `S09_PD_Borja_DatasetCodificado.csv`.

El insumo ya trae la partición entrenamiento / prueba en la columna `conjunto`, porque los
escaladores de la sesión 8 se ajustaron con ese entrenamiento. Esta sesión conserva la misma
partición y la aplica con la misma disciplina: primero se separa, después se ajusta, y el ajuste
solo ve entrenamiento. La variable objetivo que se usa para el target encoding es `fuga_cliente`.

In [1]:
import os
import re
import unicodedata
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)

ENTRADA = "S08_PD_Grupo05_DatasetEscalado.csv"
SALIDA  = "S09_PD_Borja_DatasetCodificado.csv"
OBJETIVO = "fuga_cliente"
SEMILLA = 42

df = pd.read_csv(ENTRADA, low_memory=False)

pd.DataFrame({"valor": [len(df), df.shape[1]]}, index=["filas", "columnas"])

,valor
filas,620000
columnas,40


---
# Parte 1 · Inventario, clasificación y cardinalidad

## 1.1 · Qué columnas son categóricas y cuáles solo lo parecen

No toda columna de texto es una variable categórica, y no toda columna numérica deja de serlo.
Antes de codificar se recorre columna por columna y se le asigna un rol. Solo las que resulten
nominales u ordinales entran al proceso de codificación; el resto se conserva tal cual o se excluye
con una razón explícita.

In [2]:
ROLES = {
    "id_pedido":                 ("identificador", "un valor por fila, no aporta información de comportamiento"),
    "id_cliente":                ("identificador", "152.176 clientes; codificarlo con el objetivo memorizaría al cliente"),
    "correo_cliente":            ("identificador", "casi único por fila; solo su dominio es una categoría real"),
    "fecha_pedido":              ("fecha", "es una fecha, no una categoría; se conserva para ingeniería temporal"),
    "fecha_actualizacion_stock": ("fecha", "es una fecha, no una categoría; se conserva para ingeniería temporal"),
    "canal_compra":              ("nominal", "Web, Tienda, Marketplace, App: no hay orden entre canales"),
    "metodo_pago":               ("nominal", "Transferencia, PayPal, Tarjeta, Efectivo: no hay orden entre medios"),
    "ciudad_tienda":             ("nominal", "doce ciudades; ninguna es mayor o menor que otra"),
    "codigo_postal":             ("nominal", "entero en el archivo pero es una etiqueta geográfica; 493 valores"),
    "categoria_producto":        ("nominal", "seis categorías de catálogo sin jerarquía entre ellas"),
    "producto":                  ("nominal", "24 productos del catálogo, sin orden"),
    "comentario_cliente":        ("nominal", "13 frases fijas de un catálogo de comentarios; no es texto libre"),
    "nivel_satisfaccion":        ("ordinal", "Bajo < Medio < Alto es un orden real de intensidad"),
    "nivel_lealtad":             ("ordinal", "Bronce < Plata < Oro < Platino es un orden real de nivel"),
    "conjunto":                  ("partición", "marca entrenamiento / prueba heredada de S08; se conserva"),
    "fuga_cliente":              ("objetivo", "binaria en 0 y 1; es la variable objetivo del target encoding"),
}

inventario = []
for c in df.columns:
    if c in ROLES:
        rol, razon = ROLES[c]
    elif df[c].dtype == bool:
        rol, razon = "bandera", "0 / 1 creada en sesiones previas; ya está codificada"
    elif c.endswith("_esc") or c.endswith("_original"):
        rol, razon = "numérica", "variable de comportamiento escalada o su copia de trazabilidad"
    else:
        rol, razon = "numérica", "variable de comportamiento; no se codifica"
    inventario.append({"columna": c, "tipo de dato": str(df[c].dtype),
                       "valores distintos": int(df[c].nunique()), "rol": rol, "razón": razon})

inventario = pd.DataFrame(inventario).set_index("columna")
inventario[inventario["rol"].isin(["nominal", "ordinal", "identificador", "fecha", "objetivo", "partición"])]

,tipo de dato,valores distintos,rol,razón
columna,,,,
id_pedido,object,620000,identificador,"un valor por fila, no aporta información de comportamiento"
id_cliente,object,152176,identificador,152.176 clientes; codificarlo con el objetivo memorizaría al cliente
fecha_pedido,object,836,fecha,"es una fecha, no una categoría; se conserva para ingeniería temporal"
canal_compra,object,4,nominal,"Web, Tienda, Marketplace, App: no hay orden entre canales"
metodo_pago,object,4,nominal,"Transferencia, PayPal, Tarjeta, Efectivo: no hay orden entre medios"
ciudad_tienda,object,12,nominal,doce ciudades; ninguna es mayor o menor que otra
codigo_postal,int64,493,nominal,entero en el archivo pero es una etiqueta geográfica; 493 valores
categoria_producto,object,6,nominal,seis categorías de catálogo sin jerarquía entre ellas
producto,object,24,nominal,"24 productos del catálogo, sin orden"


**Nominal u ordinal, columna por columna.** Una variable es ordinal solo si sus categorías tienen un
orden que existe en el negocio y no en el alfabeto. `nivel_satisfaccion` (Bajo, Medio, Alto) y
`nivel_lealtad` (Bronce, Plata, Oro, Platino) cumplen esa condición: pasar de un nivel al siguiente
significa más satisfacción o más lealtad. Las demás no la cumplen. Un canal de compra no es mayor
que otro, una ciudad no es mayor que otra, un producto no es mayor que otro. Imponerles un orden con
un entero (Web = 0, Tienda = 1, App = 2) inventaría una distancia que el negocio no reconoce, y
cualquier modelo que use distancias o coeficientes la tomaría por verdadera.

**`codigo_postal` es categórica aunque venga como entero.** La sesión 8 ya lo había dejado fuera del
escalado por la misma razón: 1100103 no es "mayor" que 5400101, es otra zona. Se trata como nominal
de alta cardinalidad.

**`comentario_cliente` no es texto libre.** Tiene exactamente trece valores, doce frases fijas y la
marca `sin comentario`, así que se comporta como un catálogo y no como lenguaje natural. Eso permite
codificarla como nominal de baja cardinalidad en lugar de necesitar técnicas de texto.

**Los identificadores se excluyen de la codificación, no por pereza sino por riesgo.** `id_pedido` es
único por fila y `correo_cliente` casi. `id_cliente` merece una nota aparte: tiene 152.176 valores y
uno podría tentarse a hacerle target encoding "porque es de alta cardinalidad". Sería un error grave:
81.827 de los 85.295 clientes del conjunto de prueba también aparecen en entrenamiento, así que el
promedio de fuga por cliente aprendido en entrenamiento sería, para la mayoría de las filas de prueba,
el valor de fuga de ese mismo cliente. El modelo memorizaría al cliente en lugar de aprender de su
comportamiento. La única categoría legítima que se extrae de estos campos es el dominio del correo.

**Las fechas y las banderas no entran.** Las dos fechas se conservan para una futura ingeniería de
variables temporales (mes, día de semana, antigüedad); no son categorías. Las trece banderas
booleanas ya viven en 0 y 1.

## 1.2 · Cardinalidad medida antes de decidir

La técnica se decide con la cardinalidad en la mano, no de memoria. Para cada columna nominal u
ordinal se cuenta el número de categorías, el peso de la más frecuente y cuántas categorías son
raras (menos del 0,05 % de las filas). Ese último dato es el que separa un one-hot razonable de uno
que produciría cientos de columnas casi vacías.

In [3]:
CATEGORICAS = [c for c in ROLES if ROLES[c][0] in ("nominal", "ordinal")]
UMBRAL_RARA = 0.0005   # 0,05 % de las filas

cardinalidad = []
for c in CATEGORICAS:
    vc = df[c].value_counts(normalize=True)
    cardinalidad.append({
        "columna": c, "tipo": ROLES[c][0],
        "categorías": int(vc.size),
        "más frecuente": str(vc.index[0]),
        "% más frecuente": round(100 * vc.iloc[0], 2),
        "% menos frecuente": round(100 * vc.iloc[-1], 4),
        "categorías raras (< 0,05 %)": int((vc < UMBRAL_RARA).sum()),
    })
cardinalidad = pd.DataFrame(cardinalidad).set_index("columna").sort_values("categorías")
cardinalidad

,tipo,categorías,más frecuente,% más frecuente,% menos frecuente,"categorías raras (< 0,05 %)"
columna,,,,,,
canal_compra,nominal,4,Web,25.03,24.9513,0
metodo_pago,nominal,4,Transferencia,25.05,24.9755,0
nivel_lealtad,ordinal,4,Bronce,25.06,24.9381,0
categoria_producto,nominal,6,Belleza,16.74,16.5752,0
nivel_satisfaccion,ordinal,10,Alto,28.06,0.9853,0
ciudad_tienda,nominal,12,Cartagena,8.37,8.2935,0
comentario_cliente,nominal,13,sin comentario,14.19,7.0990,0
producto,nominal,24,Parlante bluetooth,4.22,4.1232,0
codigo_postal,nominal,493,5000101,3.21,0.0002,262


In [4]:
def decidir(fila):
    if fila["tipo"] == "ordinal":
        return "ordinal encoding con orden explícito"
    if fila["categorías"] <= 30 and fila["categorías raras (< 0,05 %)"] == 0:
        return "one-hot encoding"
    return "agrupación de raras + frequency encoding + target encoding"

cardinalidad["técnica"] = cardinalidad.apply(decidir, axis=1)
cardinalidad[["tipo", "categorías", "categorías raras (< 0,05 %)", "técnica"]]

,tipo,categorías,"categorías raras (< 0,05 %)",técnica
columna,,,,
canal_compra,nominal,4,0,one-hot encoding
metodo_pago,nominal,4,0,one-hot encoding
nivel_lealtad,ordinal,4,0,ordinal encoding con orden explícito
categoria_producto,nominal,6,0,one-hot encoding
nivel_satisfaccion,ordinal,10,0,ordinal encoding con orden explícito
ciudad_tienda,nominal,12,0,one-hot encoding
comentario_cliente,nominal,13,0,one-hot encoding
producto,nominal,24,0,one-hot encoding
codigo_postal,nominal,493,262,agrupación de raras + frequency encoding + target encoding


**Lectura de la tabla.** Hay tres grupos claramente separados.

- Siete columnas nominales con entre 4 y 24 categorías, todas balanceadas (la más frecuente nunca
  supera el 26 % y la menos frecuente nunca baja del 4 %, salvo `comentario_cliente` donde
  `sin comentario` pesa 14 %). Ninguna tiene categorías raras. Para ellas el one-hot es la técnica
  correcta: produce entre 4 y 24 columnas por variable, ninguna casi vacía, y no impone orden.
  `producto` está en el límite con 24 categorías, pero como están perfectamente balanceadas el
  one-hot no genera columnas inútiles, así que se mantiene en este grupo.
- Dos columnas ordinales con 4 y 10 categorías. Las diez de `nivel_satisfaccion` son en realidad
  tres niveles escritos de distintas formas más una marca de faltante, como se ve en la sección
  siguiente.
- Una columna con 493 categorías, `codigo_postal`, de las cuales 262 son raras. Un one-hot aquí
  crearía 493 columnas, más de la mitad con menos de 310 unos entre 620.000 filas. Es la única
  variable que requiere el tratamiento de alta cardinalidad.

El dominio del correo, que se deriva en la sección 1.4, se suma al primer grupo con 6 categorías.

## 1.3 · Inconsistencia heredada en `nivel_satisfaccion`

La columna trae los tres niveles escritos en tres variantes de mayúsculas, más la marca `Sin dato`
que la sesión de imputación dejó como texto y registró en la bandera `satisfaccion_sin_dato`. Si se
codificara tal como viene, el codificador ordinal tomaría `alto`, `Alto` y `ALTO` como tres
categorías distintas y no habría manera de darles un orden coherente. Se normaliza antes de todo.

In [5]:
antes = df["nivel_satisfaccion"].value_counts(dropna=False)

df["nivel_satisfaccion"] = df["nivel_satisfaccion"].str.strip().str.capitalize()
df.loc[df["nivel_satisfaccion"] == "Sin dato", "nivel_satisfaccion"] = np.nan

despues = df["nivel_satisfaccion"].value_counts(dropna=False).rename(index={np.nan: "nulo"})

pd.concat([antes.rename("antes"), despues.rename("después")], axis=1).fillna(0).astype(int)

,antes,después
nivel_satisfaccion,,
Alto,173956,192456
Medio,173102,191514
Bajo,172448,191030
Sin dato,45000,0
alto,12391,0
medio,12262,0
bajo,12206,0
BAJO,6376,0
MEDIO,6150,0


In [6]:
pd.DataFrame({"valor": [
    int(df["nivel_satisfaccion"].isna().sum()),
    int(df["satisfaccion_sin_dato"].sum()),
    bool((df["nivel_satisfaccion"].isna() == df["satisfaccion_sin_dato"]).all()),
    sorted(df["nivel_satisfaccion"].dropna().unique().tolist()),
]}, index=["nulos tras normalizar", "bandera satisfaccion_sin_dato",
           "nulos y bandera coinciden fila a fila", "categorías que quedan"])

,valor
nulos tras normalizar,45000
bandera satisfaccion_sin_dato,45000
nulos y bandera coinciden fila a fila,True
categorías que quedan,"[Alto, Bajo, Medio]"


Las diez variantes se reducen a tres niveles limpios. Los 45.000 registros marcados como `Sin dato`
pasan a nulo, y ese nulo coincide fila a fila con la bandera `satisfaccion_sin_dato` que ya existía,
así que no se pierde información: el codificador ordinal va a dejar esas filas en nulo y la bandera
sigue diciendo por qué. Inventarles un nivel (por ejemplo, tratarlas como "Medio") sería imputar, y
la imputación es una decisión de otra sesión.

## 1.4 · Una categoría real escondida en un identificador: el dominio del correo

`correo_cliente` es casi único por fila y no se codifica. Pero su dominio (`gmail.com`,
`outlook.com`, etc.) sí es una categoría nominal con pocos valores, y es el tipo de información que
un modelo de fuga podría usar. Al extraerlo aparece un problema heredado: la columna tiene tres
formas. Correos válidos con `@`, 102.000 registros con el texto `desconocido` que la sesión de
imputación dejó marcados en `correo_sin_dato`, y 50.000 correos donde la `@` fue reemplazada por un
punto (`cliente593809.hotmail.com`). En estos últimos el dominio sigue siendo recuperable, así que
se extrae con una expresión que contempla las dos formas; los `desconocido` se conservan como una
categoría propia, coherente con su bandera.

In [7]:
correo = df["correo_cliente"].str.strip().str.lower()
tiene_arroba = correo.str.contains("@", na=False)
formato_punto = ~tiene_arroba & correo.str.match(r"^cliente\d+\.", na=False)

df["dominio_correo"] = "desconocido"
df.loc[tiene_arroba, "dominio_correo"] = correo[tiene_arroba].str.split("@").str[-1]
df.loc[formato_punto, "dominio_correo"] = correo[formato_punto].str.replace(r"^cliente\d+\.", "", regex=True)

display(pd.DataFrame({"valor": [
    int(tiene_arroba.sum()), int(formato_punto.sum()), int((correo == "desconocido").sum()),
    int(df["correo_sin_dato"].sum()),
    bool(((df["dominio_correo"] == "desconocido") == df["correo_sin_dato"]).all()),
]}, index=["correos con @", "correos con punto en lugar de @", "registros 'desconocido'",
           "bandera correo_sin_dato", "'desconocido' y la bandera coinciden fila a fila"]))

pd.DataFrame({
    "filas": df["dominio_correo"].value_counts(),
    "%": (100 * df["dominio_correo"].value_counts(normalize=True)).round(2),
})

,valor
correos con @,468000
correos con punto en lugar de @,50000
registros 'desconocido',102000
bandera correo_sin_dato,102000
'desconocido' y la bandera coinciden fila a fila,True


,filas,%
dominio_correo,,
gmail.com,104110,16.79
novamarket.com.co,103668,16.72
yahoo.es,103536,16.70
outlook.com,103474,16.69
hotmail.com,103212,16.65
desconocido,102000,16.45


Seis categorías: los cinco dominios reales más `desconocido`, que coincide fila a fila con la
bandera `correo_sin_dato`. Los 50.000 correos con punto en lugar de `@` quedan registrados como
pendiente de calidad de datos para la columna original, pero su dominio ya está recuperado y no
contamina la codificación.

---
# Parte 2 · La partición va antes que cualquier codificador que aprenda

## 2.1 · Se conserva la partición de la sesión 8

Dos de las técnicas de esta sesión aprenden de los datos: el target encoding aprende el promedio de
`fuga_cliente` por categoría y el frequency encoding aprende la frecuencia de cada categoría. El
one-hot y el ordinal, en cambio, solo aprenden la lista de categorías, y aun así se ajustan con
entrenamiento para que la lista de columnas quede fijada antes de ver la prueba.

No se vuelve a partir el dataset. La sesión 8 ya dividió 80 / 20 con `random_state = 42` y ajustó
los escaladores con ese entrenamiento. Si esta sesión hiciera una partición nueva, una fila que hoy
está en prueba podría quedar en un entrenamiento cuyo escalado ya la conoció, y la disciplina de las
dos sesiones se contradiría. La columna `conjunto` es la partición, y se respeta.

In [8]:
es_train = df["conjunto"] == "entrenamiento"
es_test  = df["conjunto"] == "prueba"

pd.DataFrame({
    "filas": [int(es_train.sum()), int(es_test.sum()), len(df)],
    "proporción": [round(es_train.mean(), 3), round(es_test.mean(), 3), 1.0],
    "tasa de fuga": [round(df.loc[es_train, OBJETIVO].mean(), 4),
                     round(df.loc[es_test, OBJETIVO].mean(), 4),
                     round(df[OBJETIVO].mean(), 4)],
}, index=["entrenamiento", "prueba", "total"])

,filas,proporción,tasa de fuga
entrenamiento,496000,0.8,0.2093
prueba,124000,0.2,0.2075
total,620000,1.0,0.2090


## 2.2 · Qué pasaría si el target encoding se calculara con todo el dataset

El argumento no debería quedarse en la teoría. Se calcula, solo para mostrarlo, el promedio de fuga
por código postal de dos maneras: con el dataset completo (la forma incorrecta) y solo con
entrenamiento (la forma correcta). Luego se mira qué le pasa a los códigos raros, que es donde la
fuga de datos hace más daño.

In [9]:
frec_train = df.loc[es_train, "codigo_postal"].value_counts()
codigos_raros = frec_train[frec_train < UMBRAL_RARA * es_train.sum()].index

media_completa = df.groupby("codigo_postal")[OBJETIVO].mean()                    # incorrecto: ve la prueba
media_train    = df.loc[es_train].groupby("codigo_postal")[OBJETIVO].mean()      # correcto
media_test     = df.loc[es_test].groupby("codigo_postal")[OBJETIVO].mean()       # lo que habría que predecir
n_test         = df.loc[es_test, "codigo_postal"].value_counts()

comparacion = pd.DataFrame({
    "filas train": frec_train,
    "fuga train": media_train.round(3),
    "filas test": n_test,
    "fuga test": media_test.round(3),
    "fuga con dataset completo": media_completa.round(3),
}).loc[codigos_raros].dropna(subset=["filas test"]).sort_values("filas train")

comparacion.head(12)

,filas train,fuga train,filas test,fuga test,fuga con dataset completo
codigo_postal,,,,,
7600194,1,1.0,2.0,0.500,0.667
1700187,2,0.5,1.0,1.000,0.667
6800181,2,0.0,1.0,0.000,0.000
6800194,2,0.0,3.0,0.000,0.000
1700185,2,0.5,1.0,0.000,0.333
6800180,2,0.0,1.0,0.000,0.000
6800183,2,0.5,1.0,0.000,0.333
1700183,2,0.0,2.0,0.000,0.000
1300194,2,0.0,1.0,0.000,0.000


In [10]:
raros_con_test = comparacion.index
pd.DataFrame({"valor": [
    len(codigos_raros),
    int(frec_train[codigos_raros].sum()),
    round(100 * frec_train[codigos_raros].sum() / es_train.sum(), 2),
    int((media_train[codigos_raros] == 0).sum()),
    int((media_train[codigos_raros] == 1).sum()),
    len(raros_con_test),
    round(float(np.abs(media_completa[raros_con_test] - media_test[raros_con_test]).mean()), 4),
    round(float(np.abs(media_train[raros_con_test] - media_test[raros_con_test]).mean()), 4),
]}, index=["códigos raros en entrenamiento", "filas de entrenamiento en códigos raros",
           "% del entrenamiento en códigos raros",
           "códigos raros con fuga train exactamente 0", "códigos raros con fuga train exactamente 1",
           "códigos raros que también aparecen en prueba",
           "error medio en prueba usando el dataset completo",
           "error medio en prueba usando solo entrenamiento"])

,valor
códigos raros en entrenamiento,263.0000
filas de entrenamiento en códigos raros,12578.0000
% del entrenamiento en códigos raros,2.5400
códigos raros con fuga train exactamente 0,67.0000
códigos raros con fuga train exactamente 1,2.0000
códigos raros que también aparecen en prueba,203.0000
error medio en prueba usando el dataset completo,0.1439
error medio en prueba usando solo entrenamiento,0.1953


**Lo que muestra la tabla.** El promedio calculado con el dataset completo se parece mucho más al
promedio de prueba que el calculado solo con entrenamiento. Eso no es una virtud: es la prueba de
que ya vio las filas de prueba. Con códigos de tres o cuatro filas, la media "completa" incorpora
directamente el valor de fuga de las filas que después se querrían predecir, y el modelo parecería
acertar sin haber aprendido nada.

También se ve el segundo problema de los códigos raros, independiente de la fuga: 67 códigos tienen
fuga de entrenamiento exactamente 0 y 2 tienen exactamente 1, porque con dos o tres filas cualquier
promedio es extremo. Un target encoding ingenuo trasladaría esos ceros y unos al modelo como si
fueran certezas. Por eso, en la Parte 5, los códigos raros se agrupan antes de codificar y el
codificador aplica suavizado hacia la media global.

---
# Parte 3 · One-hot para las nominales de baja cardinalidad

## 3.1 · Ajuste con entrenamiento, transformación de ambos conjuntos

Siete columnas: `canal_compra`, `metodo_pago`, `categoria_producto`, `ciudad_tienda`,
`comentario_cliente`, `dominio_correo` y `producto`. El codificador se ajusta con entrenamiento para
fijar la lista de categorías y el orden de las columnas; como las siete son balanceadas, todas sus
categorías aparecen en entrenamiento y la prueba no trae ninguna desconocida. Si la trajera,
`handle_unknown = "ignore"` la dejaría como una fila de ceros en lugar de romper la transformación.

No se elimina la primera categoría de cada variable. La "trampa de las variables ficticias" solo
afecta a modelos lineales sin regularización, que no son el destino declarado de este dataset;
conservar todas las columnas mantiene cada categoría legible y evita que una de ellas quede implícita.
Si más adelante se ajusta una regresión lineal sin regularización, basta con descartar una columna
por grupo en ese momento.

In [11]:
NOMINALES_OH = ["canal_compra", "metodo_pago", "categoria_producto", "ciudad_tienda",
                "comentario_cliente", "dominio_correo", "producto"]

def limpiar_nombre(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()
    return s

onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.uint8)
onehot.fit(df.loc[es_train, NOMINALES_OH])                       # ajuste: solo entrenamiento

nombres_oh = [limpiar_nombre(n) for n in onehot.get_feature_names_out(NOMINALES_OH)]
X_oh = pd.DataFrame(onehot.transform(df[NOMINALES_OH]),           # transformación: ambos conjuntos
                    columns=nombres_oh, index=df.index)

resumen_oh = pd.DataFrame({
    "categorías aprendidas en entrenamiento": [len(c) for c in onehot.categories_],
    "categorías presentes en prueba": [df.loc[es_test, c].nunique() for c in NOMINALES_OH],
    "columnas generadas": [len(c) for c in onehot.categories_],
}, index=NOMINALES_OH)
resumen_oh.loc["total"] = resumen_oh.sum()
resumen_oh

,categorías aprendidas en entrenamiento,categorías presentes en prueba,columnas generadas
canal_compra,4,4,4
metodo_pago,4,4,4
categoria_producto,6,6,6
ciudad_tienda,12,12,12
comentario_cliente,13,13,13
dominio_correo,6,6,6
producto,24,24,24
total,69,69,69


## 3.2 · Verificación

Cada fila debe tener exactamente un 1 dentro del grupo de columnas de cada variable, tanto en
entrenamiento como en prueba, y los nombres de columna deben ser legibles y sin caracteres que
compliquen su uso.

In [12]:
verif_oh = []
for c, cats in zip(NOMINALES_OH, onehot.categories_):
    cols = [limpiar_nombre(f"{c}_{k}") for k in cats]
    suma_train = X_oh.loc[es_train, cols].sum(axis=1)
    suma_test  = X_oh.loc[es_test, cols].sum(axis=1)
    verif_oh.append({
        "variable": c, "columnas": len(cols),
        "un solo 1 por fila (train)": bool((suma_train == 1).all()),
        "un solo 1 por fila (test)": bool((suma_test == 1).all()),
        "ejemplo de columna": cols[0],
    })
pd.DataFrame(verif_oh).set_index("variable")

,columnas,un solo 1 por fila (train),un solo 1 por fila (test),ejemplo de columna
variable,,,,
canal_compra,4,True,True,canal_compra_app
metodo_pago,4,True,True,metodo_pago_efectivo
categoria_producto,6,True,True,categoria_producto_belleza
ciudad_tienda,12,True,True,ciudad_tienda_barranquilla
comentario_cliente,13,True,True,comentario_cliente_buena_atencion_pero_demoro_mucho
dominio_correo,6,True,True,dominio_correo_desconocido
producto,24,True,True,producto_audifonos_inalambricos


In [13]:
pd.concat([df[NOMINALES_OH[:3]], X_oh.iloc[:, :14]], axis=1).head(5)

,canal_compra,metodo_pago,categoria_producto,canal_compra_app,canal_compra_marketplace,canal_compra_tienda,canal_compra_web,metodo_pago_efectivo,metodo_pago_paypal,metodo_pago_tarjeta,metodo_pago_transferencia,categoria_producto_belleza,categoria_producto_deportes,categoria_producto_electronica,categoria_producto_hogar,categoria_producto_jugueteria,categoria_producto_moda
0,Tienda,Efectivo,Moda,0,0,1,0,1,0,0,0,0,0,0,0,0,1
1,Web,Efectivo,Electrónica,0,0,0,1,1,0,0,0,0,0,1,0,0,0
2,Tienda,PayPal,Juguetería,0,0,1,0,0,1,0,0,0,0,0,0,1,0
3,App,Transferencia,Deportes,1,0,0,0,0,0,0,1,0,1,0,0,0,0
4,Web,Efectivo,Hogar,0,0,0,1,1,0,0,0,0,0,0,1,0,0


Las siete variables cumplen la regla de un único 1 por fila en los dos conjuntos: no hay filas
sin categoría ni filas con dos. Los nombres quedan en minúsculas, sin tildes ni espacios
(`comentario_cliente_llego_incompleto_el_pedido` en lugar de `comentario_cliente_Llegó incompleto
el pedido`), lo que evita problemas al leer el CSV desde cualquier herramienta. Se generan 69
columnas binarias en total.

---
# Parte 4 · Ordinal con el orden real, no el alfabético

## 4.1 · El orden se declara, no se deduce

Un `OrdinalEncoder` sin categorías declaradas ordena alfabéticamente. Para `nivel_lealtad` eso
daría Bronce = 0, Oro = 1, Plata = 2, Platino = 3, poniendo Oro por debajo de Plata; para
`nivel_satisfaccion` daría Alto = 0, Bajo = 1, Medio = 2, con Alto como el nivel más bajo. Ambos
órdenes son falsos y cualquier modelo los tomaría por verdaderos. Por eso el orden se escribe
explícitamente.

In [14]:
ORDENES = {
    "nivel_lealtad":      ["Bronce", "Plata", "Oro", "Platino"],
    "nivel_satisfaccion": ["Bajo", "Medio", "Alto"],
}
ORDINALES = list(ORDENES)

ordinal = OrdinalEncoder(categories=[ORDENES[c] for c in ORDINALES],
                         handle_unknown="use_encoded_value", unknown_value=-1, dtype=float)
ordinal.fit(df.loc[es_train, ORDINALES])                                          # ajuste: solo entrenamiento

X_ord = pd.DataFrame(ordinal.transform(df[ORDINALES]),                            # transformación: ambos
                     columns=[c + "_ord" for c in ORDINALES], index=df.index)

for c in ORDINALES:                       # un nulo no es una categoría desconocida: se conserva como nulo
    X_ord.loc[df[c].isna(), c + "_ord"] = np.nan

mapeo = []
for c in ORDINALES:
    for i, cat in enumerate(ORDENES[c]):
        mapeo.append({"variable": c, "categoría": cat, "código asignado": i,
                      "código si fuera alfabético": sorted(ORDENES[c]).index(cat)})
pd.DataFrame(mapeo).set_index(["variable", "categoría"])

código asignado  código si fuera alfabético
variable           categoría                                             
nivel_lealtad      Bronce                   0                           0
                   Plata                    1                           2
                   Oro                      2                           1
                   Platino                  3                           3
nivel_satisfaccion Bajo                     0                           1
                   Medio                    1                           2
                   Alto                     2                           0

## 4.2 · Verificación de que el orden es el correcto

El orden declarado es una decisión de negocio, pero hay una forma de comprobar que es coherente con
los datos: si Bronce < Plata < Oro < Platino es un orden real de lealtad, la tasa de fuga debería
descender de forma monótona a lo largo de él. Lo mismo para Bajo < Medio < Alto en satisfacción. Se
calcula solo con entrenamiento.

In [15]:
verif_ord = []
for c in ORDINALES:
    tasa = df.loc[es_train].groupby(X_ord.loc[es_train, c + "_ord"])[OBJETIVO].mean()
    for cod, cat in enumerate(ORDENES[c]):
        verif_ord.append({"variable": c, "código": cod, "categoría": cat,
                          "filas train": int((X_ord.loc[es_train, c + "_ord"] == cod).sum()),
                          "tasa de fuga (train)": round(tasa.loc[cod], 4)})
verif_ord = pd.DataFrame(verif_ord).set_index(["variable", "código"])

monotonia = {c: bool(verif_ord.loc[c, "tasa de fuga (train)"].is_monotonic_decreasing) for c in ORDINALES}
display(verif_ord)
pd.DataFrame({"la fuga desciende con el nivel": monotonia})

categoría  filas train  tasa de fuga (train)
variable           código                                             
nivel_lealtad      0         Bronce       124331                0.3408
                   1          Plata       123948                0.2368
                   2            Oro       124211                0.1589
                   3        Platino       123510                0.1000
nivel_satisfaccion 0           Bajo       152876                0.3735
                   1          Medio       153373                0.1679
                   2           Alto       153725                0.0829

,la fuga desciende con el nivel
nivel_lealtad,True
nivel_satisfaccion,True


In [16]:
pd.DataFrame({
    "nulos en la columna original": [int(df[c].isna().sum()) for c in ORDINALES],
    "nulos en la columna codificada": [int(X_ord[c + "_ord"].isna().sum()) for c in ORDINALES],
    "valores -1 (categoría desconocida)": [int((X_ord[c + "_ord"] == -1).sum()) for c in ORDINALES],
}, index=ORDINALES)

,nulos en la columna original,nulos en la columna codificada,valores -1 (categoría desconocida)
nivel_lealtad,0,0,0
nivel_satisfaccion,45000,45000,0


La tasa de fuga baja de 34,1 % en Bronce a 10,0 % en Platino, y de 37,4 % en Bajo a 8,3 % en Alto,
sin ningún retroceso intermedio. Con el orden alfabético, Oro (15,9 %) habría quedado codificado por
debajo de Plata (23,7 %) y la relación se habría roto. El orden declarado es el que los datos
confirman.

Los 45.000 nulos de `nivel_satisfaccion` siguen siendo nulos en `nivel_satisfaccion_ord`, con la
bandera `satisfaccion_sin_dato` señalándolos. Hay que hacerlo de forma explícita: con una lista de
categorías declarada, el codificador trata el nulo como una categoría desconocida y le pondría el
código reservado -1, que un modelo leería como "un nivel por debajo de Bajo". No aparece ningún -1
en el resultado: la prueba no trae categorías que el entrenamiento no conociera.

---
# Parte 5 · Alta cardinalidad: `codigo_postal`

## 5.1 · Agrupación de categorías raras, con umbral aprendido en entrenamiento

493 códigos, de los cuales 263 tienen menos del 0,05 % de las filas de entrenamiento (menos de
248 filas), y 185 tienen diez filas o menos. Esos códigos no tienen suficiente evidencia para
estimar nada por separado: cualquier estadístico que se les calcule es ruido. Se agrupan en una
categoría `Otro`. La lista de códigos raros se decide mirando solo entrenamiento y se aplica igual a
los dos conjuntos.

In [17]:
df["codigo_postal_agrupado"] = df["codigo_postal"].astype(str)
df.loc[df["codigo_postal"].isin(codigos_raros), "codigo_postal_agrupado"] = "Otro"

pd.DataFrame({"valor": [
    int(df.loc[es_train, "codigo_postal"].nunique()),
    len(codigos_raros),
    int(df.loc[es_train, "codigo_postal_agrupado"].nunique()),
    int((df.loc[es_train, "codigo_postal_agrupado"] == "Otro").sum()),
    int((df.loc[es_test, "codigo_postal_agrupado"] == "Otro").sum()),
    int(len(set(df.loc[es_test, "codigo_postal"]) - set(df.loc[es_train, "codigo_postal"]))),
]}, index=["códigos distintos en entrenamiento", "códigos raros agrupados en 'Otro'",
           "categorías tras agrupar", "filas 'Otro' en entrenamiento", "filas 'Otro' en prueba",
           "códigos de prueba que no existen en entrenamiento"])

,valor
códigos distintos en entrenamiento,493
códigos raros agrupados en 'Otro',263
categorías tras agrupar,231
filas 'Otro' en entrenamiento,12578
filas 'Otro' en prueba,3218
códigos de prueba que no existen en entrenamiento,0


## 5.2 · Frequency encoding

La frecuencia de cada código en entrenamiento se convierte en una variable numérica. Captura algo
que el target encoding no captura: si un código postal es una zona de muchos pedidos o de pocos. Se
calcula sobre el código original, no sobre el agrupado, para conservar esa información también
dentro de los raros. Un código que no estuviera en entrenamiento recibiría frecuencia 0.

In [18]:
frecuencia = df.loc[es_train, "codigo_postal"].value_counts(normalize=True)   # aprendida solo en entrenamiento
df["codigo_postal_freq"] = df["codigo_postal"].map(frecuencia).fillna(0.0)    # aplicada a ambos

pd.DataFrame({
    "entrenamiento": df.loc[es_train, "codigo_postal_freq"].describe().round(5),
    "prueba": df.loc[es_test, "codigo_postal_freq"].describe().round(5),
})

,entrenamiento,prueba
count,496000.00000,124000.00000
mean,0.01326,0.01320
std,0.01125,0.01124
min,0.00000,0.00000
25%,0.00315,0.00315
50%,0.00875,0.00873
75%,0.02614,0.02614
max,0.03202,0.03202


## 5.3 · Target encoding, ajustado solo con entrenamiento y con suavizado

Se usa `TargetEncoder` de scikit-learn sobre la versión agrupada. Tres decisiones cuentan aquí:

1. **Se ajusta únicamente con entrenamiento.** El codificador guarda un promedio de fuga por
   categoría calculado con las filas de entrenamiento. La prueba se transforma con esos promedios y
   nunca aporta su propio valor de `fuga_cliente`.
2. **Dentro del entrenamiento se usa validación cruzada interna.** `fit_transform` sobre
   entrenamiento divide las filas en 5 bloques y codifica cada bloque con promedios calculados en
   los otros 4. Sin esto, cada fila de entrenamiento se codificaría con un promedio que incluye su
   propio `fuga_cliente`, que es otra forma de fuga, más sutil, y la que hace que el target encoding
   ingenuo sobreajuste. Para la prueba se usa `transform`, que aplica los promedios del
   entrenamiento completo.
3. **Suavizado hacia la media global.** Con `smooth = "auto"` el promedio de una categoría con
   pocas filas se acerca a la tasa global de fuga en proporción a su poca evidencia. Es la segunda
   defensa contra los ceros y unos de la Parte 2.

In [19]:
te = TargetEncoder(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=SEMILLA)

X_te = pd.Series(np.nan, index=df.index, name="codigo_postal_te")
X_te.loc[es_train] = te.fit_transform(df.loc[es_train, ["codigo_postal_agrupado"]],   # ajuste + cross fitting: solo entrenamiento
                                      df.loc[es_train, OBJETIVO]).ravel()
X_te.loc[es_test]  = te.transform(df.loc[es_test, ["codigo_postal_agrupado"]]).ravel()  # transformación: prueba con promedios de entrenamiento
df["codigo_postal_te"] = X_te

pd.DataFrame({
    "entrenamiento": df.loc[es_train, "codigo_postal_te"].describe().round(4),
    "prueba": df.loc[es_test, "codigo_postal_te"].describe().round(4),
})

,entrenamiento,prueba
count,496000.0000,124000.0000
mean,0.2093,0.2093
std,0.0099,0.0089
min,0.1383,0.1606
25%,0.2048,0.2053
50%,0.2092,0.2090
75%,0.2130,0.2128
max,0.2718,0.2541


## 5.4 · Verificación de que no hubo fuga

La comprobación correcta no es mirar la distribución, sino comprobar de dónde salieron los números
que el codificador guardó. Se reconstruye a mano, solo con entrenamiento, el promedio suavizado de
tres categorías (la más frecuente, una intermedia y `Otro`) y se compara con lo que el codificador
aplicó a la prueba. Además se verifica que el promedio guardado para cada categoría coincide con el
de entrenamiento y difiere del que se obtendría con el dataset completo.

In [20]:
categorias_te = te.categories_[0]
guardado = pd.Series(te.encodings_[0], index=categorias_te)
media_global_train = df.loc[es_train, OBJETIVO].mean()

por_cat = df.loc[es_train].groupby("codigo_postal_agrupado")[OBJETIVO].agg(["mean", "size", "var"])
por_cat_completo = df.groupby("codigo_postal_agrupado")[OBJETIVO].mean()

ejemplos = [por_cat["size"].idxmax(), por_cat["size"].sort_values().index[len(por_cat) // 2], "Otro"]
tabla_verif = pd.DataFrame({
    "filas train": por_cat.loc[ejemplos, "size"].astype(int),
    "fuga train sin suavizar": por_cat.loc[ejemplos, "mean"].round(4),
    "fuga dataset completo": por_cat_completo.loc[ejemplos].round(4),
    "valor guardado por el codificador": guardado.loc[ejemplos].round(4),
    "valor aplicado en prueba": [round(df.loc[es_test & (df["codigo_postal_agrupado"] == k), "codigo_postal_te"].iloc[0], 4)
                                 for k in ejemplos],
}, index=ejemplos)
tabla_verif.index.name = "categoría"
tabla_verif

,filas train,fuga train sin suavizar,fuga dataset completo,valor guardado por el codificador,valor aplicado en prueba
categoría,,,,,
5000101,15883,0.2027,0.2042,0.2027,0.2027
1700111,962,0.1996,0.2017,0.1996,0.1996
Otro,12578,0.2128,0.2133,0.2128,0.2128


In [21]:
comparacion_global = pd.DataFrame({
    "guardado": guardado, "media train": por_cat["mean"], "media completo": por_cat_completo,
}).dropna()
pd.DataFrame({"valor": [
    round(float(te.target_mean_), 4), round(float(media_global_train), 4),
    round(float(np.abs(comparacion_global["guardado"] - comparacion_global["media train"]).max()), 4),
    round(float(np.abs(comparacion_global["guardado"] - comparacion_global["media completo"]).max()), 4),
    int((np.abs(comparacion_global["guardado"] - comparacion_global["media train"])
         < np.abs(comparacion_global["guardado"] - comparacion_global["media completo"])).sum()),
    len(comparacion_global),
    bool(df.loc[es_test].groupby("codigo_postal_agrupado")["codigo_postal_te"].nunique().eq(1).all()),
]}, index=["media global guardada por el codificador", "media global de entrenamiento",
           "mayor diferencia entre valor guardado y media de entrenamiento",
           "mayor diferencia entre valor guardado y media del dataset completo",
           "categorías cuyo valor guardado está más cerca de train que del completo",
           "categorías evaluadas",
           "en prueba, cada categoría recibe un único valor (no depende de su fuga)"])

,valor
media global guardada por el codificador,0.2093
media global de entrenamiento,0.2093
mayor diferencia entre valor guardado y media de entrenamiento,0.0001
mayor diferencia entre valor guardado y media del dataset completo,0.0214
categorías cuyo valor guardado está más cerca de train que del completo,231
categorías evaluadas,231
"en prueba, cada categoría recibe un único valor (no depende de su fuga)",True


**Lectura.** La media global que guarda el codificador es exactamente la de entrenamiento. Para cada
categoría, el valor guardado es la media de entrenamiento movida ligeramente hacia esa media global
por el suavizado, y en la gran mayoría de las categorías queda más cerca de la media de
entrenamiento que de la media del dataset completo, lo que confirma que nunca vio la prueba. En
prueba, todas las filas de una misma categoría reciben el mismo número, que es lo que debe pasar
cuando el valor no depende del `fuga_cliente` de esas filas.

Con eso quedan cubiertas las dos formas de fuga: la prueba no participa del ajuste, y dentro del
entrenamiento ninguna fila se codifica con su propio objetivo gracias al cross fitting.

**Una nota honesta sobre la señal.** Entre los 230 códigos con suficiente evidencia, la tasa de fuga
se mueve entre 16 % y 25 %, con desviación de 1,5 puntos. La variable codificada es legítima y
está bien construida, pero su poder predictivo es modesto; la mayor parte de la variación de fuga
en este dataset está en `nivel_satisfaccion` y `nivel_lealtad`, no en la geografía. Eso no cambia
el procedimiento, pero conviene decirlo para que nadie espere de esta columna más de lo que puede dar.

---
# Parte 6 · Consolidación y exportación

Se arma el dataset final con las columnas numéricas, banderas, identificadores, fechas y la
partición tal como venían, más las 69 columnas one-hot, las 2 ordinales y las 2 derivadas de
`codigo_postal` (frecuencia y target encoding). Las columnas de texto que ya quedaron codificadas se retiran: dejarlas junto a su
versión codificada es la forma más fácil de que alguien las use por error en un modelo, y su
contenido se recupera con las tablas de mapeo de este notebook. Los identificadores y las fechas se
conservan porque no son variables de modelo sino de trazabilidad y de ingeniería futura.

In [22]:
TEXTO_CODIFICADO = NOMINALES_OH + ORDINALES + ["codigo_postal", "codigo_postal_agrupado"]
CONSERVAR_TEXTO  = ["id_pedido", "id_cliente", "correo_cliente", "fecha_pedido",
                    "fecha_actualizacion_stock", "conjunto"]

base = df.drop(columns=TEXTO_CODIFICADO)
final = pd.concat([base, X_oh, X_ord], axis=1)

# el objetivo y la partición al final, para que queden a la vista
final = final[[c for c in final.columns if c not in (OBJETIVO, "conjunto")] + [OBJETIVO, "conjunto"]]

texto_restante = [c for c in final.columns if final[c].dtype == object]
pd.DataFrame({"valor": [
    df.shape[1], len(X_oh.columns), len(X_ord.columns), 2, len(TEXTO_CODIFICADO), final.shape[1],
    ", ".join(texto_restante),
    bool(set(texto_restante) <= set(CONSERVAR_TEXTO)),
]}, index=["columnas de entrada (tras normalizar y derivar)", "columnas one-hot agregadas",
           "columnas ordinales agregadas", "columnas derivadas de codigo_postal (freq y te)",
           "columnas de texto retiradas", "columnas finales",
           "columnas de texto que quedan", "todas son identificadores, fechas o la partición"])

,valor
columnas de entrada (tras normalizar y derivar),44
columnas one-hot agregadas,69
columnas ordinales agregadas,2
columnas derivadas de codigo_postal (freq y te),2
columnas de texto retiradas,11
columnas finales,104
columnas de texto que quedan,"id_pedido, id_cliente, fecha_pedido, correo_cliente, fecha_actualizacion_stock, conjunto"
"todas son identificadores, fechas o la partición",True


In [23]:
final.to_csv(SALIDA, index=False)
verificacion = pd.read_csv(SALIDA, low_memory=False)

pd.DataFrame({"valor": [
    SALIDA, f"{os.path.getsize(SALIDA) / 1048576:.1f} MB",
    len(df), len(final), len(verificacion),
    int((final["conjunto"] == "entrenamiento").sum()), int((final["conjunto"] == "prueba").sum()),
    len(final) - len(df),
    int((final["conjunto"] == "entrenamiento").sum() + (final["conjunto"] == "prueba").sum() - len(final)),
    final.shape[1], verificacion.shape[1],
    int(verificacion.select_dtypes(include=object).shape[1]),
]}, index=["archivo", "tamaño", "filas de entrada", "filas finales", "filas releídas del CSV",
           "filas de entrenamiento", "filas de prueba",
           "conciliación de filas (debe ser 0)", "conciliación train + test (debe ser 0)",
           "columnas finales", "columnas releídas", "columnas de texto en el CSV releído"])

,valor
archivo,S09_PD_Borja_DatasetCodificado.csv
tamaño,266.8 MB
filas de entrada,620000
filas finales,620000
filas releídas del CSV,620000
filas de entrenamiento,496000
filas de prueba,124000
conciliación de filas (debe ser 0),0
conciliación train + test (debe ser 0),0
columnas finales,104


In [24]:
pd.concat([verificacion[["id_pedido", "nivel_lealtad_ord", "nivel_satisfaccion_ord",
                         "codigo_postal_freq", "codigo_postal_te", "canal_compra_app",
                         "canal_compra_web", "dominio_correo_gmail_com", "fuga_cliente", "conjunto"]].head(6)], axis=1)

,id_pedido,nivel_lealtad_ord,nivel_satisfaccion_ord,codigo_postal_freq,codigo_postal_te,canal_compra_app,canal_compra_web,dominio_correo_gmail_com,fuga_cliente,conjunto
0,P505195,3.0,0.0,0.006280,0.210914,0,0,0,0,prueba
1,P677836,0.0,1.0,0.030268,0.210076,0,1,0,0,entrenamiento
2,P387869,2.0,0.0,0.029681,0.213490,0,0,0,0,prueba
3,P481069,3.0,1.0,0.008367,0.202432,1,0,1,0,entrenamiento
4,P247920,1.0,2.0,0.001585,0.175658,0,1,0,0,entrenamiento
5,P465514,3.0,2.0,0.010798,0.203051,0,0,0,0,entrenamiento


Las dos filas de conciliación dan cero: el archivo tiene las mismas 620.000 filas del insumo, la suma
de entrenamiento y prueba reconstruye el total, y las únicas columnas de texto que sobreviven son
los tres identificadores, las dos fechas y la partición.

**Pendientes que esta sesión no resuelve y deja registrados.** Los nulos de `nivel_satisfaccion_ord`
(45.000) y de las variables numéricas heredadas de sesiones previas siguen sin imputar, de forma
deliberada, con sus banderas. Las dos fechas siguen como texto a la espera de una sesión de
ingeniería de variables temporales. `id_cliente` queda sin codificar y no debería entrar a un modelo
como está; si en el futuro se quisiera aprovechar el historial del cliente, el camino es construir
agregados por cliente calculados únicamente con pedidos anteriores a cada fila, nunca un target
encoding del identificador.